*Notebook 08 in `notebooks/arcs/`. Parameters (pre-registered): `analysis/arcs/config.yaml`; decisions and run notes: `analysis/arcs/LOG.md`. The static page `site/arcs/index.html` is built from the same saved outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/arcs -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.arcs import s08_prediction as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 8 · Second method: does the arc help predict profit?

Step 7 asked whether the arc is *associated* with profit once the controls are held fixed. This step asks a
more practical question: if we already know a film's genre, decade, budget and scene length, does also
knowing its arc help **predict** the profit of films the model has not seen?

**How.**
1. We split the films into 5 folds, stratified by genre × decade cell. We fit an ordinary least-squares model
   on 4 folds and predict the 5th, until every film has an out-of-fold prediction.
2. We do that 10 times with different splits.
3. We compare two models: the **baseline** (cells, log budget, scene length) and **baseline + D1 + D2**. The
   gain is the difference in out-of-fold R².
4. Compared to what:
   - the range of the gain over the 10 repeats;
   - a permutation null of 200 runs with the outcome shuffled (pre-registered);
   - an exploratory null that shuffles only D1 and D2 across films, keeping the baseline's own signal.

We say the arc "adds predictive value" only if the gain is positive in every repeat **and** beyond the
permutation null. A gap that sits inside the repeat range is reported as no gain.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

**Table.** Out-of-fold R² for both models, the gain, and the permutation p-values.

In [ ]:
display(step.tables(res)["results"])

,"out-of-fold R² of log(gross/budget), mean over 10 repeats [min, max]",value
0,baseline: cells + log budget + scene length,"0.1480 [0.1402, 0.1576]"
1,baseline + D1 + D2,"0.1460 [0.1373, 0.1565]"
2,gain from the arc,"-0.0021 [-0.0042, 0.0005]"
3,"permutation p, outcome permuted (pre-registered)",0.224
4,"permutation p, D1 and D2 permuted (exploratory)",0.358
5,decision,does not add predictive value beyond the baseline


**Figure.** The real gain (blue line; blue band = range over repeats) against the two permutation nulls.

In [ ]:
step.figures(res)["gain"].show()

**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

Out of sample, the baseline explains R² = 0.148 of log(gross/budget) (range over repeats 0.140 to 0.158). Adding D1 and D2 gives 0.146, a gain of -0.0021 (range -0.0042 to +0.0005; permutation p = 0.224 with the outcome permuted, 0.358 with the descriptors permuted). The arc does not add predictive value beyond the baseline: the gain is not positive in every repeat. We do not write that it "beats" the baseline.